# 2.2 — Configuration, entraînement et régularisation

Modèle retenu : **YOLOv8m** (25.9M paramètres)  
Dataset : SH17 — 6 479 images train / 1 458 images val / 17 classes  
GPU : Tesla P100-PCIE-16GB

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

DATASET_ROOT = Path("/root/Projet_Image/SH17dataset")
RUNS_DIR     = Path("/root/Projet_Image/runs")

print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

assert (DATASET_ROOT / 'sh17.yaml').exists(), "sh17.yaml introuvable"
print("sh17.yaml trouvé")

## 1. Chargement du modèle

In [ ]:
model = YOLO('yolov8m.pt')
print(f"Modèle chargé : {sum(p.numel() for p in model.model.parameters()):,} paramètres")

## 2. Entraînement

- `freeze=10` : backbone gelé (features COCO génériques)
- `cos_lr=True` + `warmup_epochs=3` : scheduler cosinus avec montée progressive
- `label_smoothing=0.1` : atténue la surconfiance sur les classes majoritaires (déséquilibre 118.9×)
- `patience=20` : early stopping, `best.pt` conservé automatiquement

In [ ]:
results = model.train(
    data            = str(DATASET_ROOT / 'sh17.yaml'),

    # Durée
    epochs          = 100,
    patience        = 20,

    # Résolution et batch
    imgsz           = 640,
    batch           = 16,
    device          = 0,

    # Transfer learning — gèle les 10 premières couches du backbone
    freeze          = 10,

    # Optimizer
    optimizer       = 'SGD',
    lr0             = 0.01,
    lrf             = 0.01,
    momentum        = 0.937,
    weight_decay    = 0.0005,

    # Scheduler cosine + warmup
    cos_lr          = True,
    warmup_epochs   = 3,
    warmup_momentum = 0.8,

    # Régularisation
    label_smoothing = 0.1,

    # Augmentations — traduction des concepts visualisés en 1.4 vers les paramètres ultralytics
    hsv_h           = 0.015,   # variation teinte
    hsv_s           = 0.7,     # variation saturation
    hsv_v           = 0.4,     # variation luminosité (contre-jour, nuit, éblouissement)
    fliplr          = 0.5,     # flip horizontal
    flipud          = 0.0,     # flip vertical désactivé (irréaliste)
    degrees         = 12.0,    # rotation ±12°
    translate       = 0.1,     # translation ±10%
    scale           = 0.5,     # changement d'échelle
    perspective     = 0.0005,  # vue plongeante
    mosaic          = 1.0,     # mosaic 4 images
    erasing         = 0.4,     # cutout / occlusion partielle

    # Sauvegarde
    project         = str(RUNS_DIR),
    name            = 'yolov8m_epi',
    save            = True,
    plots           = True,
)

## 3. Résultats

In [ ]:
from IPython.display import Image as IPImage, display

run_dir = RUNS_DIR / 'yolov8m_epi'

for plot in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    p = run_dir / plot
    if p.exists():
        print(f"--- {plot} ---")
        display(IPImage(str(p)))

In [ ]:
!pip install pandas -q

In [ ]:
import pandas as pd

csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    best = df.loc[df['metrics/mAP50(B)'].idxmax()]
    print(f"Meilleure epoch       : {int(best['epoch'])}")
    print(f"mAP50 val             : {best['metrics/mAP50(B)']:.3f}")
    print(f"mAP50-95 val          : {best['metrics/mAP50-95(B)']:.3f}")
    print(f"Precision val         : {best['metrics/precision(B)']:.3f}")
    print(f"Recall val            : {best['metrics/recall(B)']:.3f}")